In [1]:
!pip install -q \
  transformers \
  datasets \
  peft \
  accelerate \
  bitsandbytes \
  sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.1 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#from datasets import load_dataset

#dataset = load_dataset("json", data_files="fine_tune.jsonl")
#dataset


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 29294
    })
})

In [4]:
#loading base model Tamil LLaMA 7B
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "abhinand/tamil-llama-7b-base-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

pytorch_model-00001-of-00007.bin:   0%|          | 0.00/1.92G [00:00<?, ?B/s]

pytorch_model-00007-of-00007.bin:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

...

pytorch_model-00006-of-00007.bin:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 8,388,608 || all params: 6,877,523,968 || trainable%: 0.1220


In [ ]:
#importing instruction-response training data

from datasets import load_dataset
dataset = load_dataset("json", data_files="fine_tune.jsonl")["train"]


In [5]:
#defining example format (optional)
def format_example(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""
    return {"text": prompt}

dataset = dataset.map(format_example)


Map:   0%|          | 0/29294 [00:00<?, ? examples/s]

In [6]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)


Map:   0%|          | 0/29294 [00:00<?, ? examples/s]

In [7]:
#Setting up the training parameters

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/tamil_poetry_lora",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   

    num_train_epochs=2,              
    learning_rate=2e-4,

    fp16=True,
    logging_steps=50,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    report_to="none"
)


In [10]:
# SFT initialization

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)


In [ ]:
#training call

trainer.train()


Step,Training Loss
50,13.570400
100,12.856100
150,12.747000
200,12.737500
250,12.747000
300,12.748600
350,12.735800
400,12.734500
450,12.738300
500,12.741700


In [ ]:
#Continuing training from savepoint

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/tamil_poetry_lora/checkpoint-3000")


Step,Training Loss
3050,12.695700
3100,12.715100
3150,12.708000
3200,12.704000
3250,12.716000
3300,12.716500
3350,12.706200
3400,12.709400
3450,12.705900
3500,12.708900


In [ ]:
# to keep runtime alive (optional)

import time
while True:
    time.sleep(60)


In [2]:
#saving trained model

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "abhinand/tamil-llama-7b-base-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/tamil_poetry_lora/checkpoint-4500"
)

model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

pytorch_model-00006-of-00007.bin:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(47957, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [2]:
#verifying if the trained adapter exists

model.save_pretrained("/content/drive/MyDrive/tamil_poetry_lora_final")
tokenizer.save_pretrained("/content/drive/MyDrive/tamil_poetry_lora_final")


('/content/drive/MyDrive/tamil_poetry_lora_final/tokenizer_config.json',
 '/content/drive/MyDrive/tamil_poetry_lora_final/special_tokens_map.json',
 '/content/drive/MyDrive/tamil_poetry_lora_final/tokenizer.model',
 '/content/drive/MyDrive/tamil_poetry_lora_final/added_tokens.json',
 '/content/drive/MyDrive/tamil_poetry_lora_final/tokenizer.json')

In [1]:
#!unzip "/content/drive/MyDrive/tamil_poetry_ai.zip" -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0056.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0056.html  
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0543.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0543.html  
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0113.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0113.html  
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0453_02.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0453_02.html  
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0810.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0810.html  
  inflating: /content/tamil_poetry_ai/data/unicode_text/pmuni0443_02.html  
  inflating: /content/__MACOSX/tamil_poetry_ai/data/unicode_text/._pmuni0443_02.ht

In [1]:
import sys
sys.path.append("/content/tamil_poetry_ai")

In [2]:
from src.generator import generate_poem

 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA adapter...
Model ready!



In [3]:
#import os
#os.makedirs("/content/offload", exist_ok=True)


In [6]:
poem = generate_poem(
        theme="அறம்",
        poet="ஒளவையார்",
        num_lines=2
    )

print("\n FINAL POEM:\n")
for line in poem:
    print(line)


🎯 Generating 2-line poem (BALANCED MODE)
Poet: ஒளவையார் | Theme: அறம்

--- Line 1/2 ---
  [ 1] 'தந்தைவழித் தந்தை வழிப் பேரர்க்கு  இன்ப'Theme
  [ 2] 'பற்றோடு ஒப்பர் ஒருவர்க்கு அழகு இல்லை  சொல'Theme
  [ 3] 'தீயாதே தீர்த்தடித் தீயதன்பம்' Incomplete
  [ 4] 'இரந்தார்க்கு இல்லை உறவுடையார்க்குத்' Incomplete
  [ 5] 'ஆடைதீயாம் ஆவிலே ஆடைதரும்' Incomplete
  [ 6] 'புல்லறிவாப் பொறுக்கும் தமிழே'OK

--- Line 2/2 ---
  [ 1] 'பழிந்தார்க் கரும்பிடுவார் கண்டீர்' Incomplete
  [ 2] 'நமக்குற்றம் மறப்பதேல் நன்றாக'Theme
  [ 3] 'தற்றோர்க்கு யாரே பிறத்தல் நன்று  நல்லற'OK


📝 FINAL POEM:
1. புல்லறிவாப் பொறுக்கும் தமிழே
2. தற்றோர்க்கு யாரே பிறத்தல் நன்று  நல்லற


 FINAL POEM:

புல்லறிவாப் பொறுக்கும் தமிழே
தற்றோர்க்கு யாரே பிறத்தல் நன்று  நல்லற


In [5]:
poem= generate_poem(
    theme="காதல்",
    poet="பாரதிதாசன்",
    num_lines=3
)

print("\n FINAL POEM:\n")
for line in poem:
    print(line)



🎯 Generating 3-line poem (BALANCED MODE)
Poet: பாரதிதாசன் | Theme: காதல்

--- Line 1/3 ---
  [ 1] 'இதற்குமுன்நான் நீதான்  என்று கேட்டதே இல்லையாம்' Incomplete
  [ 2] 'இரண்டுபேர் அன்றோ ஆத்தாயி டம்மாள்' Incomplete
  [ 3] 'உரைக்கும் போதுதான் அவர் கள்பால் அன்புடைவார்  என்று'OK

--- Line 2/3 ---
  [ 1] 'அரசினிலே செந்தாமரைக் கொண்டியில் சோற்று'Theme
  [ 2] 'சிறுமை முதலில் வரலாற்றுப் பூதம் வரும்படி'Theme
  [ 3] 'பசும்போ தாகின் தமிழர்க்குச் செம்மை'Theme
  [ 4] 'ஏழ்மை யெல்லாம் ஓங்குமறந்து வாழ்வேன் என்றார்' Incomplete
  [ 5] 'அந்நாள் நானும் என் மனைவ னுந்தான் என்றான்  அவன் கண்கள் இரு' 9w
  [ 6] 'பாங்குறோ எனநினைத்தான் என்றன் கணவன்  தன்னில்' Incomplete
  [ 7] 'நாட்டினிலே வந்து நின்றோம் அங்கே அன்றோம் என்றன்' Incomplete
  [ 8] 'மாடமறவர் என்பவர்கள் தாம்எங்கும் வாழ்கின்றார்' Incomplete
  [ 9] 'அருமை மகிழ்ந்து வணங்கினார்  அப்போது அங்கு'OK

--- Line 3/3 ---
  [ 1] 'விளக்கை வைத்தவாறே  சம்மறித்திட' 3w
  [ 2] 'மறிக்கவும் தாமும் வரு தந்தான்  என்றனள்' Incomplete
  [ 3] 'எண்சீர் விருத்தம் ஒவ்வொன்றும் சிறந்தன'Th